<a href="https://colab.research.google.com/github/arunkumar-am/Integration/blob/main/Jaro_Session_on_Agents_Intro_openai_function_calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from openai import OpenAI
import json
import os
from dotenv import load_dotenv

# loads from .env by default
# OPENAI_API_KEY=your_api_key_here

load_dotenv("api_key.env")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

client = OpenAI()

# Simple API calling

In [ ]:
messages = [{"role": "user", "content": "What's the weather like in Paris today?"}]

completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
)

dict(completion.choices[0].message)

{'content': "I'm sorry, but I can't provide real-time weather updates. For the most current weather in Paris, please check a reliable weather service or app.",
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'audio': None,
 'function_call': None,
 'tool_calls': None}

In [ ]:
print(completion.choices[0].message.content)

I'm sorry, but I can't provide real-time weather updates. For the most current weather in Paris, please check a reliable weather service or app.


# OPEN ROUTER IMPLEMENTATION

In [ ]:
from openai import OpenAI
import requests
import os

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

# 1. Initialize the OpenAI client pointing to OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"), # Replace with your API key
    default_headers={
        "HTTP-Referer": "https://your-site-url.com", # Optional: for OpenRouter rankings
        "X-Title": "Your App Name",                  # Optional: for OpenRouter rankings
    }
)

tool_1 = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for provided coordinates in celsius.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True
    }
}

tools = [tool_1]

messages = [{"role": "user", "content": "What's the weather like in Paris today?"}]

completion = client.chat.completions.create(
    # 2. Use an OpenRouter model alias
    model="openai/gpt-4o",
    messages=messages,
    tools=tools,
)

print(dict(completion.choices[0].message))

# Single Tool Use

In [ ]:
import requests

def get_weather(latitude, longitude):
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

In [ ]:
tool_1 = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current temperature for provided coordinates in celsius.",
        "parameters": {
            "type": "object",
            "properties": {
                "latitude": {"type": "number"},
                "longitude": {"type": "number"}
            },
            "required": ["latitude", "longitude"],
            "additionalProperties": False
        },
        "strict": True
    }
}

tools = [tool_1]

In [ ]:
messages = [{"role": "user", "content": "What's the weather like in Paris today?"}]

completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools=tools,
)

dict(completion.choices[0].message)

{'content': None,
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'audio': None,
 'function_call': None,
 'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_j6OIN4MjQrn0pkwMt23chU4U', function=Function(arguments='{"latitude": 48.8566, "longitude": 2.3522}', name='get_weather'), type='function')]}

In [ ]:
tool_call = completion.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

result = get_weather(args["latitude"], args["longitude"])
print(result)

21.1


In [ ]:
messages.append(completion.choices[0].message)  # append model's function call message
messages.append({                               # append result message
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": str(result)
})

completion_2 = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools=tools,
)

print(completion_2.choices[0].message)

ChatCompletionMessage(content='The current temperature in Paris is approximately 21.1°C.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


# Multiple Tool Use

In [ ]:
tool_2 = {
    "type": "function",
    "function": {
        "name": "send_email",
        "description": "Send an email to a given recipient with a subject and message.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "The recipient email address."
                },
                "subject": {
                    "type": "string",
                    "description": "Email subject line."
                },
                "body": {
                    "type": "string",
                    "description": "Body of the email message."
                }
            },
            "required": [
                "to",
                "subject",
                "body"
            ],
            "additionalProperties": False
        },
        "strict": True
    }
}

In [ ]:
tools = [tool_1,tool_2]

In [ ]:
messages = [{"role": "user", "content": "Find the weather in Paris today and email this information to Bob."}]

completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools = tools,
)

dict(completion.choices[0].message)

{'content': None,
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'audio': None,
 'function_call': None,
 'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_NllKSeSruwbCAaXeo3qWBM7Z', function=Function(arguments='{"latitude": 48.8566, "longitude": 2.3522}', name='get_weather'), type='function'),
  ChatCompletionMessageFunctionToolCall(id='call_gqHHjgGCxPmEwixIW0pCbcW9', function=Function(arguments='{"to": "Bob", "subject": "Weather in Paris Today", "body": "The weather in Paris today is expected to be clear with a temperature of approximately 20°C. Have a great day!"}', name='send_email'), type='function')]}

In [ ]:
def send_email(to, subject, body):
    # return "success"
    return f"Email successfully sent to {to} with the subject {subject}"

In [ ]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)
    if name == "send_email":
        return send_email(**args)

In [ ]:
assistant_message = completion.choices[0].message
print(assistant_message)
messages.append(assistant_message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_NllKSeSruwbCAaXeo3qWBM7Z', function=Function(arguments='{"latitude": 48.8566, "longitude": 2.3522}', name='get_weather'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_gqHHjgGCxPmEwixIW0pCbcW9', function=Function(arguments='{"to": "Bob", "subject": "Weather in Paris Today", "body": "The weather in Paris today is expected to be clear with a temperature of approximately 20°C. Have a great day!"}', name='send_email'), type='function')])


In [ ]:
for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    result = call_function(name, args)
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(result)
    })

In [ ]:
completion_2 = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools=tools,
)

In [ ]:
dict(completion_2.choices[0].message)

{'content': 'The weather in Paris today is clear with a temperature of approximately 21°C. The email has been successfully sent to Bob with this information.',
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'audio': None,
 'function_call': None,
 'tool_calls': None}

In [ ]:
messages = [{"role": "user", "content": "Find the weather in Mumbai today."}]

completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools = tools,
)

dict(completion.choices[0].message)

{'content': None,
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'audio': None,
 'function_call': None,
 'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_kkex6myVfZPNLJ4i5NHu1h9W', function=Function(arguments='{"latitude":19.076,"longitude":72.8777}', name='get_weather'), type='function')]}

In [ ]:
messages = [{"role": "user", "content": "Send email to Mala."}]

completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools = tools,
)

dict(completion.choices[0].message)

{'content': None,
 'refusal': None,
 'role': 'assistant',
 'annotations': [],
 'audio': None,
 'function_call': None,
 'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_FBOU0152K9w7KqBgF6cf386Z', function=Function(arguments='{"to":"Mala","subject":"Hello!","body":"This is a test email."}', name='send_email'), type='function')]}

In [ ]:
messages = [{"role": "user", "content": "Write an essay on the future of jobs."}]

completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    tools = tools,
)

dict(completion.choices[0].message)

{'content': 'The future of jobs is a topic of great interest and importance as it reflects the evolving landscape of the global economy, technology, and society. Over the coming decades, numerous factors such as automation, artificial intelligence, globalization, and changing demographics will significantly influence the types of employment opportunities available, the skills required, and the nature of work itself.\n\nOne of the most prominent trends shaping the future of jobs is the rise of automation and artificial intelligence (AI). Many routine and repetitive tasks are increasingly being performed by machines and algorithms, leading to the displacement of certain jobs, particularly in manufacturing, data entry, and administrative roles. However, this technological shift also creates new opportunities in emerging fields such as robotics, AI development, cybersecurity, and data science. Workers will need to adapt by acquiring new skills that complement technology rather than compete